# cap10_sequence_risk_target_cap_bc_screen_v1

Purpose: convert the previous sequence-risk diagnostic into a fold-local target-cap screen. The previous notebook identified `trend_break` as the strongest diagnostic state for the remaining `wf_2022` BC drawdown miss. This notebook tests whether a non-leaking, fold-local cap on that state can improve BC official replay before any longer PPO run.

Governance:
- Canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- Direct comparison baseline: `cap10_bc_sequence_risk_state_diagnostic_v1` and the original `cap10_joint_contract_checkpoint_drift_selector_v1` BC initialization.
- Role: `diagnose`.
- Changed layer: `model_selection` / `training_objective` / `risk_state_target` only.
- This notebook does not retain or promote an actor. It only decides whether the target-cap recipe is worth a later checkpoint/PPO screen.
- Target gate remains excess >= `+30pp` and MaxDD <= `15%` across evaluated walk-forward folds.

Anti-leakage rule:
- The feature family (`trend_break`) comes from the previous diagnostic as a mechanism hypothesis.
- The actual thresholds and cap labels are recomputed inside each replay fold's train segment from train-period feature quantiles. No 2022 threshold from the diagnostic card, and no wf_2022 2021 validation row, is reused for training labels.

External basis:
- Qlib workflow separation and replay discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO is deferred until target/reward evidence is stronger: https://arxiv.org/abs/1707.06347
- Conservative / safe policy improvement caution: https://arxiv.org/abs/2006.04779 and https://arxiv.org/abs/1712.06924
- Backtest overfit / multiple-testing discipline: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.path_diagnostics import make_backtest_daily_frame
from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import ACCOUNT, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'cap10_sequence_risk_target_cap_bc_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
SOURCE_SEQUENCE_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_bc_sequence_risk_state_diagnostic_v1'
SOURCE_CHECKPOINT_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_joint_contract_checkpoint_drift_selector_v1'
TARGET_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_foldlocal_joint_target_contract_v1'
TARGETS_PATH = TARGET_CARD / 'joint_target_contract.csv'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'

SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
BC_SLICE_ACCURACY_MIN = 0.60
TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)

CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

SPEC_BY_TARGET_FOLD = {str(spec['target_fold']): spec for spec in WALK_FORWARD_WINDOWS.values()}

CAP_CANDIDATES = [
    {
        'candidate': 'baseline_no_sequence_cap',
        'cap': None,
        'mode': 'none',
        'features': [],
        'complexity': 0,
        'description': 'Original joint contract without sequence-risk edits.',
    },
    {
        'candidate': 'slope_low_cap15',
        'cap': 0.15,
        'mode': 'any',
        'features': ['anchor_slope_180'],
        'complexity': 1,
        'description': 'Cap only the strongest prior diagnostic feature: low long-anchor slope.',
    },
    {
        'candidate': 'anchor_any_cap15',
        'cap': 0.15,
        'mode': 'any',
        'features': ['anchor_slope_180', 'anchor_gap_180'],
        'complexity': 2,
        'description': 'Cap if either long-anchor slope or anchor gap is in the weak quartile.',
    },
    {
        'candidate': 'trend_break_any_cap15',
        'cap': 0.15,
        'mode': 'any',
        'features': ['anchor_slope_180', 'anchor_gap_180', 'trend_gap_63'],
        'complexity': 3,
        'description': 'Use the full previous trend_break feature set with a 15% position cap.',
    },
    {
        'candidate': 'trend_break_any_cap10',
        'cap': 0.10,
        'mode': 'any',
        'features': ['anchor_slope_180', 'anchor_gap_180', 'trend_gap_63'],
        'complexity': 4,
        'description': 'Full trend_break state with a stricter 10% cap.',
    },
    {
        'candidate': 'trend_break_two_of_three_cap15',
        'cap': 0.15,
        'mode': 'two_of_three',
        'features': ['anchor_slope_180', 'anchor_gap_180', 'trend_gap_63'],
        'complexity': 3,
        'description': 'Require at least two trend_break feature hits before capping.',
    },
]

FEATURE_DIRECTIONS = {
    'anchor_slope_180': 'low',
    'anchor_gap_180': 'low',
    'trend_gap_63': 'high',
}


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n')


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/cap10_sequence_risk_target_cap_bc_screen_v1


In [2]:
sequence_selection = json.loads((SOURCE_SEQUENCE_CARD / 'selection.json').read_text())
checkpoint_selection = json.loads((SOURCE_CHECKPOINT_CARD / 'selection.json').read_text())
checkpoint_metrics = pd.read_csv(SOURCE_CHECKPOINT_CARD / 'checkpoint_candidate_metrics.csv')
risk_screen = pd.read_csv(SOURCE_SEQUENCE_CARD / 'risk_state_candidate_screen.csv')
targets = pd.read_csv(TARGETS_PATH)
targets['datetime'] = pd.to_datetime(targets['datetime'])
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in targets.columns:
        targets[col] = targets[col].fillna(False).astype(bool)
frame = pd.read_pickle(FRAME_PATH).sort_index()
profile = load_profile(PROFILE_NAME)

selected_risk_group = risk_screen.iloc[0].to_dict() if not risk_screen.empty else {}
source_bc = checkpoint_metrics[
    checkpoint_metrics['candidate'].eq('bc_initialization')
].copy()

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start, train_end = map(pd.Timestamp, spec['train'])
    valid_start, valid_end = map(pd.Timestamp, spec['valid'])
    test_start, test_end = map(pd.Timestamp, spec['test'])
    fold_targets = targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(train_start, train_end)]
    coverage_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'train_target_rows': int(len(fold_targets)),
        'valid_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(valid_start, valid_end)].shape[0]),
        'test_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(test_start, test_end)].shape[0]),
        'release_gate_rows': int(fold_targets.get('release_gate_state', pd.Series(False, index=fold_targets.index)).sum()),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).sum()),
        'drawdown_control_rows': int(fold_targets.get('drawdown_control_state', pd.Series(False, index=fold_targets.index)).sum()),
        'source_bc_relative_return': float(source_bc[source_bc['fold'].eq(fold_name)]['test_relative_return'].iloc[0]) if fold_name in set(source_bc['fold']) else np.nan,
        'source_bc_max_dd': float(source_bc[source_bc['fold'].eq(fold_name)]['test_max_dd'].iloc[0]) if fold_name in set(source_bc['fold']) else np.nan,
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {
    'rows': coverage.to_dict('records'),
    'sequence_source_decision': sequence_selection.get('decision'),
    'checkpoint_source_decision': checkpoint_selection.get('decision'),
    'selected_risk_group': selected_risk_group,
})

display(coverage)
display(risk_screen.head(3))


,fold,target_fold,train_start,train_end,valid_start,valid_end,test_start,test_end,train_target_rows,valid_overlap_rows,test_overlap_rows,release_gate_rows,downtrend_rows,drawdown_control_rows,source_bc_relative_return,source_bc_max_dd
0,wf_2022,wf_2022_train,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31,253,252,0,40,118,49,36.419124,18.455088
1,wf_2023_2024,wf_2023_2024_train,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31,756,0,0,40,478,49,30.501116,13.597475


,feature_group,candidate_gate,score,mean_abs_delta_z,max_abs_delta_z,flag_rate,breach_capture,pre_breach_capture,sequence_capture,non_sequence_rate,...,sequence_rows,breach_rows,pre_breach_rows,flagged_avg_pos,unflagged_avg_pos,flagged_avg_drawdown_pct,unflagged_avg_drawdown_pct,flagged_next_return_mean,unflagged_next_return_mean,feature_specs
0,trend_break,True,2.092371,0.881028,1.908566,0.564,1.000000,1.000000,0.985714,0.400000,...,70,30,33,0.166073,0.126938,10.188497,1.853297,-0.000467,-0.000874,"[{""abs_delta_z"": 1.9085659879088879, ""delta_z""..."
1,tracking_decay,True,1.961850,0.816242,1.347655,0.548,0.966667,0.939394,0.914286,0.405556,...,70,30,33,0.156633,0.139768,9.528942,2.947986,-0.000203,-0.001178,"[{""abs_delta_z"": 1.3476549592424019, ""delta_z""..."
2,volatility_stress,True,1.864461,1.239387,1.718616,0.528,0.733333,0.909091,0.800000,0.422222,...,70,30,33,0.174978,0.119961,8.399095,4.490737,-0.000324,-0.001001,"[{""abs_delta_z"": 1.7186155455380494, ""delta_z""..."


In [3]:
def effective_target_position(df: pd.DataFrame) -> pd.Series:
    teacher = pd.to_numeric(df['teacher_position'], errors='coerce').fillna(0.0)
    rebound = pd.to_numeric(df['rebound_target_position'], errors='coerce').fillna(teacher)
    risk = pd.to_numeric(df['risk_target_position'], errors='coerce').fillna(teacher)
    downtrend = df.get('is_downtrend_state', pd.Series(False, index=df.index)).fillna(False).astype(bool)
    rebound_state = df.get('is_rebound_cluster', pd.Series(False, index=df.index)).fillna(False).astype(bool)
    return pd.Series(np.where(downtrend, risk, np.where(rebound_state, rebound, teacher)), index=df.index).clip(0.0, 1.0)


def fold_local_thresholds(feature_frame: pd.DataFrame, dates: pd.Series, features: list[str]) -> dict[str, dict[str, float | str]]:
    if not features:
        return {}
    idx = pd.DatetimeIndex(pd.to_datetime(dates)).normalize()
    work = feature_frame.reindex(idx)
    thresholds: dict[str, dict[str, float | str]] = {}
    for feature in features:
        direction = FEATURE_DIRECTIONS[feature]
        values = pd.to_numeric(work[feature], errors='coerce').dropna()
        if values.empty:
            thresholds[feature] = {'direction': direction, 'threshold': float('nan'), 'quantile': 0.25 if direction == 'low' else 0.75}
            continue
        if direction == 'low':
            quantile = 0.25
            threshold = float(values.quantile(quantile))
        else:
            quantile = 0.75
            threshold = float(values.quantile(quantile))
        thresholds[feature] = {'direction': direction, 'threshold': threshold, 'quantile': quantile}
    return thresholds


def sequence_risk_mask(feature_frame: pd.DataFrame, dates: pd.Series, thresholds: dict[str, dict[str, float | str]], mode: str) -> tuple[pd.Series, pd.Series]:
    if not thresholds or mode == 'none':
        index = pd.RangeIndex(len(dates))
        return pd.Series(False, index=index), pd.Series(0, index=index, dtype=int)
    idx = pd.DatetimeIndex(pd.to_datetime(dates)).normalize()
    work = feature_frame.reindex(idx).reset_index(drop=True)
    hits = []
    for feature, spec in thresholds.items():
        threshold = float(spec['threshold'])
        values = pd.to_numeric(work[feature], errors='coerce')
        if not math.isfinite(threshold):
            hit = pd.Series(False, index=work.index)
        elif spec['direction'] == 'low':
            hit = values <= threshold
        else:
            hit = values >= threshold
        hits.append(hit.fillna(False).astype(bool))
    hit_count = pd.concat(hits, axis=1).sum(axis=1) if hits else pd.Series(0, index=work.index, dtype=int)
    if mode == 'two_of_three':
        mask = hit_count >= 2
    elif mode == 'all':
        mask = hit_count >= len(thresholds)
    else:
        mask = hit_count >= 1
    return mask.astype(bool).reset_index(drop=True), hit_count.astype(int).reset_index(drop=True)


def train_scope_group(group: pd.DataFrame, fold: str) -> pd.DataFrame:
    spec = SPEC_BY_TARGET_FOLD.get(str(fold))
    if spec is None:
        return group.copy()
    start, end = map(pd.Timestamp, spec['train'])
    return group[group['datetime'].between(start, end)].copy()


def apply_sequence_cap(base_targets: pd.DataFrame, candidate: dict[str, Any], feature_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = base_targets.copy().reset_index(drop=True)
    cap = candidate['cap']
    mode = str(candidate['mode'])
    features = list(candidate['features'])
    out['sequence_risk_state'] = False
    out['sequence_risk_feature_hits'] = 0
    out['sequence_risk_cap'] = np.nan if cap is None else float(cap)
    out['sequence_risk_candidate'] = str(candidate['candidate'])
    out['sequence_risk_thresholds_json'] = '{}'
    out['sequence_scope'] = 'outside_replay_train_segment'
    rows = []
    if cap is None:
        for fold, group in out.groupby('fold', sort=True):
            train_group = train_scope_group(group, str(fold))
            rows.append({
                'candidate': candidate['candidate'],
                'fold': fold,
                'feature_count': 0,
                'train_target_rows': int(len(train_group)),
                'all_fold_rows': int(len(group)),
                'sequence_rows': 0,
                'sequence_share': 0.0,
                'cap': np.nan,
                'mode': mode,
                'thresholds_json': '{}',
            })
            out.loc[train_group.index, 'sequence_scope'] = 'replay_train_segment'
        return out, pd.DataFrame(rows)

    for fold, group in out.groupby('fold', sort=True):
        train_group = train_scope_group(group, str(fold))
        thresholds = fold_local_thresholds(feature_frame, train_group['datetime'], features)
        mask, hit_count = sequence_risk_mask(feature_frame, train_group['datetime'], thresholds, mode)
        idx = train_group.index.to_numpy()
        out.loc[idx, 'sequence_scope'] = 'replay_train_segment'
        base_effective = effective_target_position(out.loc[idx])
        out.loc[idx, 'sequence_risk_state'] = mask.to_numpy(dtype=bool)
        out.loc[idx, 'sequence_risk_feature_hits'] = hit_count.to_numpy(dtype=int)
        out.loc[idx, 'sequence_risk_thresholds_json'] = json.dumps(json_safe(thresholds), sort_keys=True)
        active_idx = idx[mask.to_numpy(dtype=bool)]
        if len(active_idx):
            active_base = base_effective.loc[mask.to_numpy(dtype=bool)].to_numpy(dtype=float)
            active_capped = np.minimum(active_base, float(cap))
            out.loc[active_idx, 'risk_target_position'] = active_capped
            out.loc[active_idx, 'is_downtrend_state'] = True
            out.loc[active_idx, 'drawdown_control_state'] = True
            if 'source_components' in out.columns:
                out.loc[active_idx, 'source_components'] = out.loc[active_idx, 'source_components'].astype(str) + '+sequence_risk_cap'
            if 'source_component_count' in out.columns:
                out.loc[active_idx, 'source_component_count'] = pd.to_numeric(out.loc[active_idx, 'source_component_count'], errors='coerce').fillna(1).astype(int) + 1
        rows.append({
            'candidate': candidate['candidate'],
            'fold': str(fold),
            'feature_count': int(len(features)),
            'train_target_rows': int(len(train_group)),
            'all_fold_rows': int(len(group)),
            'unused_or_valid_rows': int(len(group) - len(train_group)),
            'sequence_rows': int(mask.sum()),
            'sequence_share': float(mask.mean()) if len(mask) else 0.0,
            'mean_feature_hits': float(hit_count.mean()) if len(hit_count) else 0.0,
            'cap': float(cap),
            'mode': mode,
            'thresholds_json': json.dumps(json_safe(thresholds), sort_keys=True),
            'mean_base_target_on_sequence': float(base_effective.loc[mask.to_numpy(dtype=bool)].mean()) if bool(mask.any()) else 0.0,
            'mean_capped_target_on_sequence': float(np.minimum(base_effective.loc[mask.to_numpy(dtype=bool)], float(cap)).mean()) if bool(mask.any()) else 0.0,
        })
    return out, pd.DataFrame(rows)


def action_distribution(target_frame: pd.DataFrame, candidate: str) -> pd.DataFrame:
    work = target_frame.copy()
    work['_target'] = effective_target_position(work)
    work['_action'] = np.round(work['_target'].clip(0.0, 1.0) / 0.05).astype(int)
    rows = []
    for fold, group in work.groupby('fold', sort=True):
        scoped = train_scope_group(group, str(fold))
        if scoped.empty:
            continue
        counts = scoped['_action'].value_counts(normalize=True)
        rows.append({
            'candidate': candidate,
            'fold': str(fold),
            'samples': int(len(scoped)),
            'all_fold_rows': int(len(group)),
            'unique_actions': int(scoped['_action'].nunique()),
            'dominant_action': int(counts.index[0]),
            'dominant_action_share': float(counts.iloc[0]),
            'sequence_rows': int(scoped.get('sequence_risk_state', pd.Series(False, index=scoped.index)).fillna(False).astype(bool).sum()),
            'sequence_share': float(scoped.get('sequence_risk_state', pd.Series(False, index=scoped.index)).fillna(False).astype(bool).mean()),
            'mean_target': float(scoped['_target'].mean()),
        })
    return pd.DataFrame(rows)

feature_frame = frame[[feature for feature in FEATURE_DIRECTIONS if feature in frame.columns]].copy()
cap_frames = []
cap_audits = []
action_audits = []
for candidate in CAP_CANDIDATES:
    adjusted, audit = apply_sequence_cap(targets, candidate, feature_frame)
    adjusted.to_csv(CARD / f"{candidate['candidate']}_target_contract.csv", index=False)
    cap_frames.append(adjusted)
    cap_audits.append(audit)
    action_audits.append(action_distribution(adjusted, str(candidate['candidate'])))

cap_audit = pd.concat(cap_audits, ignore_index=True, sort=False)
action_audit = pd.concat(action_audits, ignore_index=True, sort=False)
cap_audit.to_csv(CARD / 'sequence_risk_cap_audit.csv', index=False)
action_audit.to_csv(CARD / 'target_action_distribution.csv', index=False)

display(cap_audit)
display(action_audit)


,candidate,fold,feature_count,train_target_rows,all_fold_rows,sequence_rows,sequence_share,cap,mode,thresholds_json,unused_or_valid_rows,mean_feature_hits,mean_base_target_on_sequence,mean_capped_target_on_sequence
0,baseline_no_sequence_cap,wf_2022_train,0,253,505,0,0.000000,NaN,none,{},NaN,NaN,NaN,NaN
1,baseline_no_sequence_cap,wf_2023_2024_train,0,756,756,0,0.000000,NaN,none,{},NaN,NaN,NaN,NaN
2,slope_low_cap15,wf_2022_train,1,253,505,64,0.252964,0.15,any,"{""anchor_slope_180"": {""direction"": ""low"", ""qua...",252.0,0.252964,0.505560,0.150000
3,slope_low_cap15,wf_2023_2024_train,1,756,756,189,0.250000,0.15,any,"{""anchor_slope_180"": {""direction"": ""low"", ""qua...",0.0,0.250000,0.190795,0.134467
4,anchor_any_cap15,wf_2022_train,2,253,505,85,0.335968,0.15,any,"{""anchor_gap_180"": {""direction"": ""low"", ""quant...",252.0,0.505929,0.434495,0.144427
5,anchor_any_cap15,wf_2023_2024_train,2,756,756,225,0.297619,0.15,any,"{""anchor_gap_180"": {""direction"": ""low"", ""quant...",0.0,0.500000,0.201378,0.133322
6,trend_break_any_cap15,wf_2022_train,3,253,505,144,0.569170,0.15,any,"{""anchor_gap_180"": {""direction"": ""low"", ""quant...",252.0,0.758893,0.474637,0.146710
7,trend_break_any_cap15,wf_2023_2024_train,3,756,756,401,0.530423,0.15,any,"{""anchor_gap_180"": {""direction"": ""low"", ""quant...",0.0,0.750000,0.315867,0.139333
8,trend_break_any_cap10,wf_2022_train,3,253,505,144,0.569170,0.10,any,"{""anchor_gap_180"": {""direction"": ""low"", ""quant...",252.0,0.758893,0.474637,0.098794
9,trend_break_any_cap10,wf_2023_2024_train,3,756,756,401,0.530423,0.10,any,"{""anchor_gap_180"": {""direction"": ""low"", ""quant...",0.0,0.750000,0.315867,0.096316


,candidate,fold,samples,all_fold_rows,unique_actions,dominant_action,dominant_action_share,sequence_rows,sequence_share,mean_target
0,baseline_no_sequence_cap,wf_2022_train,253,505,17,4,0.134387,0,0.000000,0.419408
1,baseline_no_sequence_cap,wf_2023_2024_train,756,756,18,4,0.227513,0,0.000000,0.277530
2,slope_low_cap15,wf_2022_train,253,505,17,3,0.332016,64,0.252964,0.329464
3,slope_low_cap15,wf_2023_2024_train,756,756,18,3,0.296296,189,0.250000,0.263448
4,anchor_any_cap15,wf_2022_train,253,505,17,3,0.379447,85,0.335968,0.321954
5,anchor_any_cap15,wf_2023_2024_train,756,756,18,3,0.317460,225,0.297619,0.257275
6,trend_break_any_cap15,wf_2022_train,253,505,16,3,0.612648,144,0.569170,0.232762
7,trend_break_any_cap15,wf_2023_2024_train,756,756,16,3,0.539683,401,0.530423,0.183892
8,trend_break_any_cap10,wf_2022_train,253,505,17,2,0.545455,144,0.569170,0.205489
9,trend_break_any_cap10,wf_2023_2024_train,756,756,16,2,0.501323,401,0.530423,0.161075


In [4]:
def target_gate(metrics: dict[str, Any]) -> bool:
    return bool(float(metrics.get('relative_return', float('-inf'))) >= TARGET_RELATIVE_RETURN and float(metrics.get('max_dd', float('inf'))) <= TARGET_MAX_DD)


def slice_gate_from_metrics(slice_metrics: pd.DataFrame, *, require_sequence: bool) -> tuple[bool, dict[str, Any]]:
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    required = {
        'release_gate_state': 20,
        'is_downtrend_state': 100,
        'drawdown_control_state': 20,
    }
    if require_sequence:
        required['sequence_risk_state'] = 20
    details = {}
    gate = True
    for name, min_samples in required.items():
        if name not in indexed.index:
            details[name] = {'samples': 0, 'accuracy': float('nan'), 'gate': False, 'min_samples': min_samples}
            gate = False
            continue
        row = indexed.loc[name]
        samples = int(row.get('samples', 0))
        accuracy = float(row.get('accuracy', float('nan')))
        row_gate = bool(samples >= min_samples and math.isfinite(accuracy) and accuracy >= BC_SLICE_ACCURACY_MIN)
        details[name] = {'samples': samples, 'accuracy': accuracy, 'gate': row_gate, 'min_samples': min_samples}
        gate = gate and row_gate
    if not require_sequence and 'sequence_risk_state' in indexed.index:
        row = indexed.loc['sequence_risk_state']
        details['sequence_risk_state'] = {
            'samples': int(row.get('samples', 0)),
            'accuracy': float(row.get('accuracy', float('nan'))),
            'gate': None,
            'min_samples': 0,
        }
    return bool(gate), details


def evaluate_actor(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> tuple[dict[str, Any], dict[str, Any]]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    out['worker_return'] = float(split_metrics.get('return', float('nan')))
    out['worker_max_dd'] = float(split_metrics.get('max_dd', float('nan')))
    out['return'] = float(bt.get('return', float('nan')))
    out['sharpe'] = float(bt.get('sharpe', float('nan')))
    out['max_dd'] = float(bt.get('max_dd', float('nan')))
    out['effective_test_start'] = str(bt.get('effective_test_seg', ('', ''))[0])
    out['effective_test_end'] = str(bt.get('effective_test_seg', ('', ''))[1])
    # Keep relative_return from evaluate_split for consistency with existing checkpoint selector gates.
    out['relative_return'] = float(split_metrics.get('relative_outperformance', split_metrics.get('relative_return', float('nan'))))
    out['buy_and_hold_return'] = float(split_metrics.get('bnh_return', split_metrics.get('buy_and_hold_return', float('nan'))))
    return out, bt

init_qlib(QLIB_DATA_PATH)
fold_datasets = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    fold_datasets[fold_name] = {
        'train_df': segment_frame(dataset, 'train'),
        'valid_df': segment_frame(dataset, 'valid'),
        'test_df': segment_frame(dataset, 'test'),
        'valid_price': price_frame.loc[spec['valid'][0]:spec['valid'][1]],
        'test_price': price_frame.loc[spec['test'][0]:spec['test'][1]],
    }

candidate_rows = []
slice_rows = []
valid_rows = []
daily_paths: dict[tuple[str, str], pd.DataFrame] = {}

for candidate, adjusted in zip(CAP_CANDIDATES, cap_frames):
    candidate_name = str(candidate['candidate'])
    print('candidate', candidate_name)
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        set_seed(SEED)
        train_start, train_end = map(pd.Timestamp, spec['train'])
        test_start, test_end = map(pd.Timestamp, spec['test'])
        fold_targets = adjusted[
            adjusted['fold'].eq(spec['target_fold'])
            & adjusted['datetime'].between(train_start, train_end)
        ].copy()
        test_overlap = adjusted[
            adjusted['fold'].eq(spec['target_fold'])
            & adjusted['datetime'].between(test_start, test_end)
        ]
        if fold_targets.empty or not test_overlap.empty:
            candidate_rows.append({
                'candidate': candidate_name,
                'fold': fold_name,
                'status': 'blocked_or_missing_targets',
                'train_target_rows': int(len(fold_targets)),
                'test_overlap_rows': int(len(test_overlap)),
            })
            continue
        fit = fit_state_objective_bc_actor_from_frame(
            frame,
            targets=fold_targets,
            profile=profile,
            train_config=TRAIN_CONFIG,
            sample_weight_config=None,
            sample_size=0,
        )
        bc_dataset = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0)
        slice_metrics = evaluate_bc_actor_slice_fit(
            fit.actor,
            fit.obs,
            fit.actions,
            bc_dataset.metadata,
            slice_cols=('release_gate_state', 'is_downtrend_state', 'drawdown_control_state', 'sequence_risk_state', 'is_rebound_cluster'),
        )
        slice_gate, slice_details = slice_gate_from_metrics(slice_metrics, require_sequence=candidate['cap'] is not None)
        tagged_slices = slice_metrics.copy()
        tagged_slices.insert(0, 'candidate', candidate_name)
        tagged_slices.insert(1, 'fold', fold_name)
        slice_rows.append(tagged_slices)

        data = fold_datasets[fold_name]
        valid_metrics, _valid_bt = evaluate_actor(fit.actor, data['valid_df'], data['valid_price'], profile, spec['valid'])
        test_metrics, test_bt = evaluate_actor(fit.actor, data['test_df'], data['test_price'], profile, spec['test'])
        gate = target_gate(test_metrics)
        seq_count = int(fold_targets.get('sequence_risk_state', pd.Series(False, index=fold_targets.index)).fillna(False).astype(bool).sum())
        candidate_rows.append({
            'candidate': candidate_name,
            'fold': fold_name,
            'status': 'evaluated',
            'complexity': int(candidate['complexity']),
            'cap': np.nan if candidate['cap'] is None else float(candidate['cap']),
            'mode': str(candidate['mode']),
            'train_target_rows': int(len(fold_targets)),
            'sequence_rows': seq_count,
            'sequence_share': seq_count / max(int(len(fold_targets)), 1),
            'bc_after_accuracy': float(fit.metrics['after_accuracy']),
            'bc_after_loss': float(fit.metrics['after_loss']),
            'bc_slice_gate': slice_gate,
            'test_return': float(test_metrics['return']),
            'test_sharpe': float(test_metrics['sharpe']),
            'test_max_dd': float(test_metrics['max_dd']),
            'test_relative_return': float(test_metrics['relative_return']),
            'test_buy_and_hold_return': float(test_metrics['buy_and_hold_return']),
            'test_avg_pos': float(test_metrics.get('avg_pos', float('nan'))),
            'test_target_gate': gate,
            'valid_return': float(valid_metrics['return']),
            'valid_sharpe': float(valid_metrics['sharpe']),
            'valid_max_dd': float(valid_metrics['max_dd']),
            'valid_relative_return': float(valid_metrics['relative_return']),
            'release_gate_accuracy': float(slice_details.get('release_gate_state', {}).get('accuracy', float('nan'))),
            'downtrend_accuracy': float(slice_details.get('is_downtrend_state', {}).get('accuracy', float('nan'))),
            'drawdown_control_accuracy': float(slice_details.get('drawdown_control_state', {}).get('accuracy', float('nan'))),
            'sequence_risk_accuracy': float(slice_details.get('sequence_risk_state', {}).get('accuracy', float('nan'))),
            'sequence_risk_samples': int(slice_details.get('sequence_risk_state', {}).get('samples', 0)),
            'sequence_risk_required': bool(candidate['cap'] is not None),
        })
        feature_frame_for_daily = frame[['anchor_slope_180', 'anchor_gap_180', 'trend_gap_63']].copy()
        daily = make_backtest_daily_frame(test_bt, label=f'{candidate_name}_{fold_name}', feature_frame=feature_frame_for_daily)
        daily['datetime'] = pd.to_datetime(daily['datetime'])
        prices = data['test_price'].reindex(pd.DatetimeIndex(daily['datetime'])).ffill().bfill().to_numpy(dtype=float)
        initial_price = float(prices[0]) if len(prices) else 1.0
        daily['price'] = prices
        daily['strategy_nav'] = daily['account'].astype(float)
        daily['buyhold_nav'] = ACCOUNT * (prices / initial_price)
        daily_paths[(candidate_name, fold_name)] = daily

candidate_metrics = pd.DataFrame(candidate_rows)
if slice_rows:
    bc_slice_metrics = pd.concat(slice_rows, ignore_index=True, sort=False)
else:
    bc_slice_metrics = pd.DataFrame()
candidate_metrics.to_csv(CARD / 'bc_candidate_metrics.csv', index=False)
bc_slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)

display(candidate_metrics)


[41396:MainThread](2026-05-26 08:19:31,558) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[41396:MainThread](2026-05-26 08:19:32,425) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[41396:MainThread](2026-05-26 08:19:32,427) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[41396:MainThread](2026-05-26 08:19:34,555) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Loading data Done


[41396:MainThread](2026-05-26 08:19:34,559) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,candidate,fold,status,complexity,cap,mode,train_target_rows,sequence_rows,sequence_share,bc_after_accuracy,...,valid_return,valid_sharpe,valid_max_dd,valid_relative_return,release_gate_accuracy,downtrend_accuracy,drawdown_control_accuracy,sequence_risk_accuracy,sequence_risk_samples,sequence_risk_required
0,baseline_no_sequence_cap,wf_2022,evaluated,0,NaN,none,253,0,0.000000,0.833992,...,4.980184,0.393608,10.528546,-58.602980,1.000,0.779661,0.918367,NaN,0,False
1,baseline_no_sequence_cap,wf_2023_2024,evaluated,0,NaN,none,756,0,0.000000,0.768212,...,17.552649,0.971021,20.436342,-38.468538,0.950,0.779874,0.979592,NaN,0,False
2,slope_low_cap15,wf_2022,evaluated,1,0.15,any,253,64,0.252964,0.845850,...,9.431165,0.726417,5.858422,-61.084253,0.825,0.875000,0.896104,0.984375,64,True
3,slope_low_cap15,wf_2023_2024,evaluated,1,0.15,any,756,189,0.250000,0.700662,...,14.587444,0.708527,22.593290,-49.465866,0.425,0.705394,0.745455,0.851064,188,True
4,anchor_any_cap15,wf_2022,evaluated,2,0.15,any,253,85,0.335968,0.861660,...,14.949864,0.921972,10.970557,-65.429199,1.000,0.906250,0.894118,0.894118,85,True
5,anchor_any_cap15,wf_2023_2024,evaluated,2,0.15,any,756,225,0.297619,0.721854,...,20.882081,0.960885,17.366260,-47.464645,0.800,0.738589,0.824786,0.857143,224,True
6,trend_break_any_cap15,wf_2022,evaluated,3,0.15,any,253,144,0.569170,0.822134,...,0.675825,0.124731,11.882953,-69.239696,1.000,0.918478,0.909722,0.909722,144,True
7,trend_break_any_cap15,wf_2023_2024,evaluated,3,0.15,any,756,401,0.530423,0.735099,...,16.262394,1.316295,9.322082,-56.602846,0.800,0.810811,0.856098,0.875000,400,True
8,trend_break_any_cap10,wf_2022,evaluated,4,0.10,any,253,144,0.569170,0.869565,...,10.311890,0.703558,9.963309,-62.434936,1.000,0.972826,0.986111,0.986111,144,True
9,trend_break_any_cap10,wf_2023_2024,evaluated,4,0.10,any,756,401,0.530423,0.778808,...,19.753943,1.347404,9.232545,-59.204964,0.800,0.871224,0.902439,0.922500,400,True


In [5]:
def candidate_loss(group: pd.DataFrame) -> float:
    loss = 0.0
    if group.empty:
        return float('inf')
    complexity = float(group['complexity'].max()) if 'complexity' in group else 0.0
    for row in group.to_dict('records'):
        rel = float(row.get('test_relative_return', float('-inf')))
        dd = float(row.get('test_max_dd', float('inf')))
        loss += max(0.0, TARGET_RELATIVE_RETURN - rel) / TARGET_RELATIVE_RETURN
        loss += 2.0 * max(0.0, dd - TARGET_MAX_DD) / TARGET_MAX_DD
        if not bool(row.get('bc_slice_gate', False)):
            loss += 0.25
    loss += 0.001 * complexity
    return float(loss)

summary_rows = []
for candidate_name, group in candidate_metrics[candidate_metrics['status'].eq('evaluated')].groupby('candidate', sort=False):
    fold_count = int(len(group))
    target_pass_count = int(group['test_target_gate'].astype(bool).sum())
    slice_pass_count = int(group['bc_slice_gate'].astype(bool).sum())
    full_gate = bool(fold_count == len(WALK_FORWARD_WINDOWS) and target_pass_count == fold_count and slice_pass_count == fold_count)
    summary_rows.append({
        'candidate': candidate_name,
        'fold_count': fold_count,
        'target_pass_count': target_pass_count,
        'slice_pass_count': slice_pass_count,
        'full_gate': full_gate,
        'candidate_loss': candidate_loss(group),
        'min_relative_return': float(group['test_relative_return'].min()),
        'max_max_dd': float(group['test_max_dd'].max()),
        'mean_avg_pos': float(group['test_avg_pos'].mean()),
        'max_sequence_share': float(group['sequence_share'].max()),
        'complexity': int(group['complexity'].max()),
        'wf_2022_relative_return': float(group[group['fold'].eq('wf_2022')]['test_relative_return'].iloc[0]) if 'wf_2022' in set(group['fold']) else np.nan,
        'wf_2022_max_dd': float(group[group['fold'].eq('wf_2022')]['test_max_dd'].iloc[0]) if 'wf_2022' in set(group['fold']) else np.nan,
        'wf_2023_2024_relative_return': float(group[group['fold'].eq('wf_2023_2024')]['test_relative_return'].iloc[0]) if 'wf_2023_2024' in set(group['fold']) else np.nan,
        'wf_2023_2024_max_dd': float(group[group['fold'].eq('wf_2023_2024')]['test_max_dd'].iloc[0]) if 'wf_2023_2024' in set(group['fold']) else np.nan,
    })
summary = pd.DataFrame(summary_rows).sort_values(['full_gate', 'candidate_loss', 'complexity'], ascending=[False, True, True]).reset_index(drop=True)
summary.to_csv(CARD / 'candidate_summary.csv', index=False)

baseline_rows = summary[summary['candidate'].eq('baseline_no_sequence_cap')]
baseline = baseline_rows.iloc[0].to_dict() if not baseline_rows.empty else {}
full_gate_rows = summary[summary['full_gate'].astype(bool)] if not summary.empty else pd.DataFrame()
robust_improvement_rows = pd.DataFrame()
if baseline and not summary.empty:
    robust_improvement_rows = summary[
        summary['candidate'].ne('baseline_no_sequence_cap')
        & (summary['min_relative_return'] >= TARGET_RELATIVE_RETURN)
        & (summary['wf_2022_max_dd'] < float(baseline['wf_2022_max_dd']))
        & (summary['max_max_dd'] <= float(baseline['max_max_dd']))
    ].copy()

if not full_gate_rows.empty:
    selected = full_gate_rows.sort_values(['candidate_loss', 'complexity']).iloc[0].to_dict()
    decision = 'sequence_risk_target_cap_bc_gate_passed_no_promotion'
    next_mechanism = 'run_checkpoint_drift_selector_on_sequence_risk_target_cap_before_any_ppo_extension'
elif not robust_improvement_rows.empty:
    selected = robust_improvement_rows.sort_values(['candidate_loss', 'complexity']).iloc[0].to_dict()
    decision = 'sequence_risk_target_cap_bc_partial_improvement_no_promotion'
    next_mechanism = 'inspect_failed_fold_and_consider_reward_drawdown_drift_penalty_before_ppo'
elif baseline:
    selected = baseline
    decision = 'sequence_risk_target_cap_bc_no_robust_improvement_no_promotion'
    next_mechanism = 'do_not_use_trend_break_target_cap_directly; test reward_drawdown_drift_penalty_or_sequence_risk_state_feature'
else:
    selected = summary.iloc[0].to_dict() if not summary.empty else {}
    decision = 'sequence_risk_target_cap_bc_screen_no_evaluable_candidate_no_promotion'
    next_mechanism = 'fix_target_cap_screen_inputs_before_more_training'
selected_candidate = str(selected.get('candidate', '')) if selected else ''

# Export daily paths for the selected candidate and baseline for inspection.
for (candidate_name, fold_name), daily in daily_paths.items():
    if candidate_name in {selected_candidate, 'baseline_no_sequence_cap'}:
        daily.to_csv(CARD / f'{candidate_name}_{fold_name}_daily_path.csv', index=False)

display(summary)


,candidate,fold_count,target_pass_count,slice_pass_count,full_gate,candidate_loss,min_relative_return,max_max_dd,mean_avg_pos,max_sequence_share,complexity,wf_2022_relative_return,wf_2022_max_dd,wf_2023_2024_relative_return,wf_2023_2024_max_dd
0,baseline_no_sequence_cap,2,1,2,False,0.452857,30.501116,18.396429,0.227605,0.000000,0,36.419124,18.396429,30.501116,13.472617
1,trend_break_any_cap10,2,1,2,False,0.610798,37.779333,19.550984,0.144141,0.569170,4,68.105462,19.550984,37.779333,11.338372
2,slope_low_cap15,2,0,1,False,1.200903,37.219027,21.008695,0.196142,0.252964,1,59.461195,16.115577,37.219027,21.008695
3,trend_break_any_cap15,2,1,2,False,1.272744,39.711227,24.523079,0.149032,0.569170,3,59.343858,24.523079,39.711227,9.805398
4,trend_break_two_of_three_cap15,2,0,1,False,1.484458,41.768073,20.600116,0.195189,0.218254,3,59.739013,20.600116,41.768073,18.635817
5,anchor_any_cap15,2,0,2,False,2.347219,37.183853,24.884684,0.212158,0.335968,2,57.616373,24.884684,37.183853,22.704458


In [6]:
sidecars = [
    'coverage.csv',
    'coverage.json',
    'sequence_risk_cap_audit.csv',
    'target_action_distribution.csv',
    'bc_candidate_metrics.csv',
    'bc_slice_metrics.csv',
    'candidate_summary.csv',
    'selection.json',
    'summary.json',
]
for name in ['baseline_no_sequence_cap', selected_candidate]:
    if name:
        for fold_name in WALK_FORWARD_WINDOWS:
            path_name = f'{name}_{fold_name}_daily_path.csv'
            if (CARD / path_name).exists() and path_name not in sidecars:
                sidecars.append(path_name)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'cap10_bc_sequence_risk_state_diagnostic_v1',
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'training_objective', 'risk_state_target'],
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic BC actors',
    'promotion_blocker': 'This is a target-cap BC screen only. Promotion still requires fixed-split plus wf_2020/wf_2022/wf_2023_2024 validation and checkpoint/PPO drift review.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'selected_candidate': selected_candidate,
    'selected_full_gate': bool(selected.get('full_gate', False)) if selected else False,
    'selected_candidate_loss': selected.get('candidate_loss') if selected else None,
    'selected_min_relative_return': selected.get('min_relative_return') if selected else None,
    'selected_max_max_dd': selected.get('max_max_dd') if selected else None,
    'selected_wf_2022_relative_return': selected.get('wf_2022_relative_return') if selected else None,
    'selected_wf_2022_max_dd': selected.get('wf_2022_max_dd') if selected else None,
    'selected_wf_2023_2024_relative_return': selected.get('wf_2023_2024_relative_return') if selected else None,
    'selected_wf_2023_2024_max_dd': selected.get('wf_2023_2024_max_dd') if selected else None,
    'baseline_wf_2022_max_dd': baseline.get('wf_2022_max_dd') if baseline else None,
    'baseline_max_max_dd': baseline.get('max_max_dd') if baseline else None,
    'robust_improvement_candidate_count': int(len(robust_improvement_rows)) if 'robust_improvement_rows' in globals() else 0,
    'source_sequence_decision': sequence_selection.get('decision'),
    'source_checkpoint_decision': checkpoint_selection.get('decision'),
    'anti_leakage_note': 'Feature family came from prior diagnostic; thresholds and caps are computed only inside each replay fold train segment, and this card permits no promotion.',
    'external_sources': sequence_selection.get('external_sources', checkpoint_selection.get('external_sources', [])),
    'sidecars': sidecars,
}

summary_payload = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'coverage': coverage.to_dict('records'),
    'cap_audit': cap_audit.to_dict('records'),
    'target_action_distribution': action_audit.to_dict('records'),
    'candidate_summary': summary.to_dict('records'),
    'baseline': baseline,
    'robust_improvement_candidates': robust_improvement_rows.to_dict('records') if 'robust_improvement_rows' in globals() and not robust_improvement_rows.empty else [],
    'candidate_metrics': candidate_metrics.to_dict('records'),
    'selected': selected,
    'risk_notes': [
        'No actor is retained or promoted from this notebook.',
        'A target-cap pass would only allow checkpoint drift review; it would not prove production readiness.',
        'Because the feature family is hypothesis-driven from a 2022 diagnostic, any apparent pass still needs fixed-split plus wf_2020 coverage before promotion.',
    ],
}
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary_payload)

lines = [
    '## Conclusion',
    '',
    f"Decision: `{selection['decision']}`.",
    '',
]
if selected:
    lines.append(
        f"Selected candidate: `{selected_candidate}` with full gate `{bool(selected.get('full_gate', False))}`, "
        f"min relative `{selected.get('min_relative_return'):.2f}pp`, max MaxDD `{selected.get('max_max_dd'):.2f}%`, "
        f"loss `{selected.get('candidate_loss'):.4f}`."
    )
    lines.append('')
    lines.append(
        f"wf_2022: rel `{selected.get('wf_2022_relative_return'):.2f}pp`, MaxDD `{selected.get('wf_2022_max_dd'):.2f}%`; "
        f"wf_2023_2024: rel `{selected.get('wf_2023_2024_relative_return'):.2f}pp`, MaxDD `{selected.get('wf_2023_2024_max_dd'):.2f}%`."
    )
else:
    lines.append('No candidate was evaluated.')
lines.extend([
    '',
    'Anti-leakage note: thresholds and caps use only each replay fold train segment; 2022 diagnostic threshold values and wf_2022 2021 validation rows are not used for labels.',
    '',
    'Risk note: no runtime promotion, no retained actor, and no PPO extension is authorized directly from this notebook. A cap is not considered useful unless it improves the wf_2022 drawdown miss without breaking another evaluated fold.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `candidate_summary.csv`, `bc_candidate_metrics.csv`, `sequence_risk_cap_audit.csv`, `target_action_distribution.csv`, selected/baseline daily paths, `selection.json`, `summary.json`.',
])
display(Markdown(chr(10).join(lines)))


## Conclusion

Decision: `sequence_risk_target_cap_bc_no_robust_improvement_no_promotion`.

Selected candidate: `baseline_no_sequence_cap` with full gate `False`, min relative `30.50pp`, max MaxDD `18.40%`, loss `0.4529`.

wf_2022: rel `36.42pp`, MaxDD `18.40%`; wf_2023_2024: rel `30.50pp`, MaxDD `13.47%`.

Anti-leakage note: thresholds and caps use only each replay fold train segment; 2022 diagnostic threshold values and wf_2022 2021 validation rows are not used for labels.

Risk note: no runtime promotion, no retained actor, and no PPO extension is authorized directly from this notebook. A cap is not considered useful unless it improves the wf_2022 drawdown miss without breaking another evaluated fold.

Next action: `do_not_use_trend_break_target_cap_directly; test reward_drawdown_drift_penalty_or_sequence_risk_state_feature`.

Artifacts: `candidate_summary.csv`, `bc_candidate_metrics.csv`, `sequence_risk_cap_audit.csv`, `target_action_distribution.csv`, selected/baseline daily paths, `selection.json`, `summary.json`.